# SleepSense - Training + Export untuk Flask API
**Team CC26-PSU230 | Coding Camp 2026 DBS Foundation**

Notebook ini melatih model dan mengekspor semua artifact yang dibutuhkan Flask API:
- sleepsense_model.keras
- scaler_params.json
- feature_meta.json

In [ ]:
# Cell 1 - Install
!pip install -q scikit-learn pandas numpy matplotlib tensorboard

In [ ]:
# Cell 2 - Upload Dataset
from google.colab import files
import os
print("Upload sleepsense_model_ready.csv")
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]
print("File:", DATA_PATH)

In [ ]:
# Cell 3 - Import
import os, json, warnings, datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = "/content/sleepsense_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("TensorFlow:", tf.__version__)

In [ ]:
# Cell 4 - Preprocess
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)

DROP_COLS = ["dataset_source", "participant_id", "stress_score", "occupation"]
df = df.drop(columns=DROP_COLS)

gender_map   = {"Male": 0, "Female": 1, "Other": 2}
agegroup_map = {"13-18": 0, "19-35": 1, "36-59": 2}
df["gender"]    = df["gender"].map(gender_map).fillna(2)
df["age_group"] = df["age_group"].map(agegroup_map).fillna(1)

age_dummies = pd.get_dummies(df["age_group"].map({0:"teen",1:"young_adult",2:"adult"}), prefix="age")
df = pd.concat([df.drop("age_group", axis=1), age_dummies], axis=1)

TARGET       = "stress_risk"
FEATURE_COLS = [c for c in df.columns if c != TARGET]
X = df[FEATURE_COLS].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)

print("Features:", FEATURE_COLS)
print("Class dist: 0={} 1={}".format(int((y==0).sum()), int((y==1).sum())))

neg, pos = (y==0).sum(), (y==1).sum()
class_weight = {0: (neg+pos)/(2*neg), 1: (neg+pos)/(2*pos)}

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15/0.85, random_state=SEED, stratify=y_temp)
print("Train/Val/Test:", len(X_train), "/", len(X_val), "/", len(X_test))

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val   = scaler.transform(X_val).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

# Simpan scaler & meta
SCALER_PATH = os.path.join(OUTPUT_DIR, "scaler_params.json")
META_PATH   = os.path.join(OUTPUT_DIR, "feature_meta.json")

with open(SCALER_PATH, "w") as f:
    json.dump({"mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist(),
               "feature_cols": FEATURE_COLS}, f, indent=2)
with open(META_PATH, "w") as f:
    json.dump({"feature_cols": FEATURE_COLS, "gender_map": gender_map,
               "agegroup_map": agegroup_map}, f, indent=2)

print("Scaler & meta disimpan")

In [ ]:
# Cell 5 - Custom Components
class AttentionScaling(layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.attention_dense = layers.Dense(units, activation="sigmoid", name="attn_gate")
    def call(self, inputs):
        return inputs * self.attention_dense(inputs)
    def get_config(self):
        cfg = super().get_config()
        cfg["units"] = self.units
        return cfg

class FocalLoss(keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha
    def call(self, y_true, y_pred):
        y_true  = tf.cast(y_true, tf.float32)
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce     = -y_true*tf.math.log(y_pred) - (1-y_true)*tf.math.log(1-y_pred)
        p_t     = y_true*y_pred + (1-y_true)*(1-y_pred)
        at      = y_true*self.alpha + (1-y_true)*(1-self.alpha)
        return tf.reduce_mean(at * tf.pow(1.0-p_t, self.gamma) * bce)
    def get_config(self):
        cfg = super().get_config()
        cfg.update({"gamma": self.gamma, "alpha": self.alpha})
        return cfg

class EarlyStoppingWithLog(keras.callbacks.Callback):
    def __init__(self, patience=10, **kwargs):
        super().__init__(**kwargs)
        self.patience = patience
        self.best_auc = 0.0
        self.wait = 0
        self.best_weights = None
        self.history_log = []
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_auc = logs.get("val_auc", 0)
        self.history_log.append({"epoch": epoch+1, **{k: round(float(v),5) for k,v in logs.items()}})
        if val_auc > self.best_auc:
            self.best_auc = val_auc
            self.best_weights = self.model.get_weights()
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
                self.model.set_weights(self.best_weights)
                print(f"EarlyStopping epoch {epoch+1}, best AUC={self.best_auc:.4f}")
    def on_train_end(self, logs=None):
        log_path = os.path.join(OUTPUT_DIR, "training_log.json")
        with open(log_path, "w") as f:
            json.dump(self.history_log, f, indent=2)
        print("Log disimpan:", log_path)

print("Custom components OK")

In [ ]:
# Cell 6 - Build Model
n_features = X_train.shape[1]

def build_model(n_features, dropout=0.3):
    inputs = keras.Input(shape=(n_features,), name="features")
    x  = layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4), name="dense_1")(inputs)
    x  = layers.BatchNormalization(name="bn_1")(x)
    x  = AttentionScaling(128, name="attention_1")(x)
    x  = layers.Dropout(dropout, name="drop_1")(x)
    x2 = layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4), name="dense_2")(x)
    x2 = layers.BatchNormalization(name="bn_2")(x2)
    x2 = layers.Dropout(dropout, name="drop_2")(x2)
    x3 = layers.Dense(32, activation="relu", name="dense_3")(x2)
    x3 = layers.BatchNormalization(name="bn_3")(x3)
    out = layers.Dense(1, activation="sigmoid", name="stress_risk")(x3)
    return keras.Model(inputs=inputs, outputs=out, name="SleepSense_StressRisk")

model = build_model(n_features)
model.summary()

In [ ]:
# Cell 7 - Training
EPOCHS     = 100
BATCH_SIZE = 256

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=FocalLoss(gamma=2.0, alpha=0.35),
    metrics=[keras.metrics.BinaryAccuracy(name="accuracy"),
             keras.metrics.AUC(name="auc"),
             keras.metrics.Precision(name="precision"),
             keras.metrics.Recall(name="recall")]
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=[
        EarlyStoppingWithLog(patience=12),
        keras.callbacks.ReduceLROnPlateau(monitor="val_auc", factor=0.5, patience=5, mode="max", verbose=1),
    ],
    verbose=1
)

In [ ]:
# Cell 8 - Evaluasi
from sklearn.metrics import mean_absolute_error

y_pred_prob = model.predict(X_test, verbose=0).flatten()
y_pred      = (y_pred_prob >= 0.5).astype(int)
acc = (y_pred == y_test).mean()
auc = roc_auc_score(y_test, y_pred_prob)
f1  = f1_score(y_test, y_pred)

print("=" * 40)
print(f"Accuracy : {acc:.4f}  {'PASS' if acc >= 0.85 else 'FAIL'} (target >= 0.85)")
print(f"AUC-ROC  : {auc:.4f}")
print(f"F1-Score : {f1:.4f}")
print("=" * 40)
print(classification_report(y_test, y_pred, target_names=["No Risk","At Risk"]))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="Train"); axes[0].plot(history.history["val_accuracy"], label="Val")
axes[0].axhline(0.85, color="red", ls="--", label="Target 85%")
axes[0].set_title("Accuracy"); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(history.history["auc"], label="Train"); axes[1].plot(history.history["val_auc"], label="Val")
axes[1].set_title("AUC-ROC"); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=120); plt.show()

In [ ]:
# Cell 9 - Simpan Model untuk Flask API
MODEL_PATH = os.path.join(OUTPUT_DIR, "sleepsense_model.keras")
model.save(MODEL_PATH)
print("Model disimpan:", MODEL_PATH)
print()
print("File yang siap untuk Flask API:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size//1024} KB)")

In [ ]:
# Cell 10 - Download semua artifact untuk Flask API
import shutil
from google.colab import files

zip_path = "/content/sleepsense_flask_models"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
print("Mendownload sleepsense_flask_models.zip ...")
files.download(zip_path + ".zip")
print()
print("Letakkan isi ZIP ini ke folder models/ di project Flask kamu:")
print("  sleepsense_model.keras  -> models/sleepsense_model.keras")
print("  scaler_params.json     -> models/scaler_params.json")
print("  feature_meta.json      -> models/feature_meta.json")